# Suite VLGDEF — Verilog definitions

`Definitions.define(unit)` gives the entities a source text declares, in its scopes, as IEEE 1800 resolves
names; `referents` tells what a name refers to.

In [ ]:
from mbse.Programs.Framework.Syntax import walk
from mbse.Programs.Verilog import Definitions as D, SystemVerilog2023, Syntax as S

SOURCES = "../../conformance/sources"


def define(text):
    unit = SystemVerilog2023.parse(text)
    return unit, D.define(unit)


def name(spelling):
    return S.Identifier(spelling=spelling)


def names(unit, spelling):
    return [n for n in walk(unit) if isinstance(n, S.NameExpression) and isinstance(n.name, S.Identifier)
            and n.name.spelling == spelling]

## VLGDEF-01 · Every entity of the conformance source, with its kind and qualified name

In [ ]:
unit = SystemVerilog2023.parse(open(f"{SOURCES}/systemverilog2023.sv", encoding="utf-8").read(), include_paths=[SOURCES])
program = D.define(unit)
entities = [str(e) for e in program.entities()]
assert entities[:7] == ["<variable ambient>", "<package logger_pkg>", "<parameter logger_pkg::FIELDS>",
                        "<localparam logger_pkg::BATTERY_LOW>", "<enumerator logger_pkg::IDLE>",
                        "<enumerator logger_pkg::SAMPLE>", "<enumerator logger_pkg::SEND>"]
assert "<argument logger_pkg::in_range::raw>" in entities and "<block sampler.counter>" in entities
assert "<localparam sampler.four.HALF>" in entities and "<instance sampler.u_fifo>" in entities
unresolved = sorted({n.name.spelling for n in walk(unit) if isinstance(n, S.NameExpression)
                     and isinstance(n.name, S.Identifier) and not D.referents(program, n)})
assert unresolved == ["FPGA", "SIMULATION", "TRACE", "option", "tb"], unresolved  # macros in an `ifdef condition, a
# covergroup's built-in `option`, and an outside module
print(len(entities), "entities")

## VLGDEF-02 · Imports: explicit ones alias, wildcard ones are found where nothing nearer is

In [ ]:
unit, program = define("""package p; parameter A = 1; parameter B = 2; parameter C = 3; endpackage
package q; parameter C = 4; endpackage
module m; import p::A; import q::*; import p::missing; import nowhere::*; import nowhere::Y;
parameter B = 5;
initial x = A + B + C + p::B + nowhere::X + p::B2;
endmodule""")
a = names(unit, "A")[0]
assert [str(e) for e in D.referents(program, a)] == ["<parameter p::A>"]
assert [str(e) for e in D.referents(program, names(unit, "B")[0])] == ["<parameter m.B>"]
assert [str(e) for e in D.referents(program, names(unit, "C")[0])] == ["<parameter q::C>"]
scoped = [n for n in walk(unit) if isinstance(n, S.ScopedName)]
assert [str(e) for e in D.referents(program, scoped[0])] == ["<parameter p::B>"]
assert D.referents(program, scoped[1]) == [] and D.referents(program, scoped[2]) == []
missing = [e for e in program.entities() if e.name == "missing"][0]
assert missing.kind == "import" and missing.target is None
print("ok")

## VLGDEF-03 · Ports, parameters, types, enumerators, functions and blocks, each in its scope

In [ ]:
unit, program = define("""module m #(parameter type T = int, parameter N) (a, b);
input a; output b; typedef enum {X, Y} e_t;
genvar g;
function int f(int v); begin : inner int t; t = v; end return v; endfunction
initial begin int w; for (int i = 0; i < 2; i++) w = i; foreach (q[j]) w = j; for (;;) ; end
for (g = 0; g < 2; g++) begin : gen end
endmodule""")
entities = [str(e) for e in program.entities()]
assert entities[:7] == ["<module m>", "<type parameter m.T>", "<parameter m.N>", "<port m.a>", "<port m.b>", "<enumerator m.X>",
                        "<enumerator m.Y>"]
port = [e for e in program.entities() if e.name == "a"][0]
assert len(port.declarations) == 2 and isinstance(port.definition, S.PortReference)
assert "<function m.f>" in entities and "<argument m.f.v>" in entities and "<block m.f.inner>" in entities
assert "<variable m.f.inner.t>" in entities and "<block m.gen>" in entities and "<genvar m.g>" in entities
i = names(unit, "i")[0]
assert [e.kind for e in D.referents(program, i)] == ["variable"]
assert [e.kind for e in D.referents(program, names(unit, "j")[0])] == ["variable"]
assert D.referents(program, names(unit, "q")[0]) == []
declared = [n for n in walk(unit) if isinstance(n, S.Identifier) and n.spelling == "w"][0]
assert [str(e) for e in D.referents(program, declared)] == ["<variable w>"]
assert D.referents(program, S.Identifier(spelling="z")) == []
print("ok")

## VLGDEF-04 · Interfaces, modports, instances and out-of-block names

In [ ]:
unit, program = define("""interface bus; logic d; modport mp (input d); endinterface
module top; bus b(); sub u (.p(b.d)); endmodule
function int c::f(); endfunction""")
entities = [str(e) for e in program.entities()]
assert "<modport bus.mp>" in entities and "<instance top.b>" in entities and "<instance top.u>" in entities
assert not any("f" == e.name for e in program.entities() if e.kind == "function" and e.name is not None)
print("ok")

## VLGDEF-05 · Classes: members, bases, constructors and methods defined outside their class

In [ ]:
unit, program = define("""class base; int x; extern function void f(int a); function new(); endfunction endclass
class derived extends base #(1); function void g(); y = x; endfunction endclass
interface class i1; parameter int P = 1; endclass
interface class i2 extends i1; endclass
function void base::f(int a); x = a + i2::P; endfunction
function void nowhere::h(); endfunction
function void base::missing(); endfunction
package p; class c; static int s; endclass endpackage
class from_package extends p::c; endclass
class from_nowhere extends unknown; endclass
class reader; function void r(); y = p::c::s + p::q::s + q::r::s + p::c::missing::s; endfunction endclass
""")
entities = [str(e) for e in program.entities()]
assert entities[:6] == ["<class base>", "<variable base::x>", "<function base::f>", "<argument base::f::a>",
                        "<function base::new>", "<class derived>"]
f = [e for e in program.entities() if e.name == "f"][0]
assert len(f.declarations) == 2 and f.definition is unit.items[4] and f.scope.parent.owner.name == "base"
assert [str(e) for e in D.referents(program, names(unit, "x")[0])] == ["<variable base::x>"]  # through derived's base
assert [str(e) for e in D.referents(program, names(unit, "x")[1])] == ["<variable base::x>"]
assert [str(e) for e in D.referents(program, names(unit, "a")[0])] == ["<argument base::f::a>"]
assert D.referents(program, names(unit, "y")[0]) == []
scoped = [n for n in walk(unit) if isinstance(n, S.ScopedName)]
assert [str(e) for e in D.referents(program, scoped[1])] == ["<parameter i1::P>"]  # through i2's interfaces
assert [str(e) for e in D.referents(program, scoped[0])] == ["<function base::f>"]
assert D.referents(program, scoped[2]) == [] and D.referents(program, scoped[3]) == []  # nowhere::h, base::missing
nested = [n for n in scoped if isinstance(n.name, S.ScopedName)]
assert [str(e) for e in D.referents(program, nested[0])] == ["<variable p::c::s>"]
assert [D.referents(program, n) for n in nested[1:]] == [[]] * 4  # no q in p, no q, no missing in c (twice)
classes = {e.name: e for e in program.entities() if e.kind == "class"}
assert classes["from_package"].scope.bases == [classes["c"].scope] and classes["from_nowhere"].scope.bases == []
handmade = S.SourceText(items=[S.FunctionDeclaration(name=S.ScopedName(scope=name("p"), name=S.ScopedName(
    scope=name("c"), name=name("f"))))])
assert [e.kind for e in D.define(handmade).entities()] == []  # a qualified name slang does not read: not resolved
assert [str(e) for e in D.referents(program, S.ScopedName(scope=name("base"), name=name("x")))] == ["<variable base::x>"]
print("ok")


## VLGDEF-06 · Constraints, their loops, `local::` and array methods' iterators

In [ ]:
unit, program = define("""class c; rand int x, y; int q[]; extern constraint e; pure constraint p;
constraint k { foreach (q[i]) q[i] < x; }
function void f(int y); ok = randomize() with { x < local::y; }; r = q.sum() with (item * 2); r = q.find(v) with (v > y); q.sort with (item.x);
endfunction endclass
constraint c::e { x != y; }
constraint nowhere::e { }
""")
entities = [str(e) for e in program.entities()]
assert entities[:7] == ["<class c>", "<variable c::x>", "<variable c::y>", "<variable c::q>", "<constraint c::e>",
                        "<constraint c::p>", "<constraint c::k>"]
e = [e for e in program.entities() if e.name == "e"][0]
assert len(e.declarations) == 2 and e.definition is unit.items[1]
assert [str(x) for x in D.referents(program, names(unit, "i")[0])] == ["<variable i>"]
local = [n for n in walk(unit) if isinstance(n, S.NameExpression) and isinstance(n.name, S.LocalName)][0]
assert [str(x) for x in D.referents(program, local)] == ["<argument c::f::y>"]
assert [str(x) for x in D.referents(program, names(unit, "item")[0])] == ["<variable item>"]
assert [str(x) for x in D.referents(program, names(unit, "item")[1])] == ["<variable item>"]  # q.sort's
assert [str(x) for x in D.referents(program, names(unit, "v")[0])] == ["<variable v>"]
assert [str(x) for x in D.referents(program, names(unit, "y")[-1])] == ["<variable c::y>"]  # in c::e, the class's
print("ok")


## VLGDEF-07 · Properties, sequences, `let`s, clocking blocks and labels

In [ ]:
unit, program = define("""module m; logic clk, a, b, rst;
default clocking cb @(posedge clk); input #1 a; output b2 = b; endclocking
default clocking @(negedge clk); input a; endclocking
sequence s(x, int y = 1); int v; (a, v = x) ##1 b ##(y) 1; endsequence
property p(sequence q); disable iff (rst) q |=> a; endproperty
let twice(n) = n * 2;
chk: assert property (@(posedge clk) p(s(a)));
initial begin lab: a = twice(1); cb.b2 <= 1; end
default clocking cb;
endmodule""")
entities = [str(e) for e in program.entities()]
for entity in ["<clocking m.cb>", "<clockvar m.cb.a>", "<clockvar m.cb.b2>", "<sequence m.s>", "<argument m.s.x>",
               "<argument m.s.y>", "<variable m.s.v>", "<property m.p>", "<argument m.p.q>", "<let m.twice>",
               "<argument m.twice.n>", "<label m.chk>", "<label lab>"]:
    assert entity in entities, entity
assert [str(e) for e in D.referents(program, names(unit, "x")[0])] == ["<argument m.s.x>"]
assert [str(e) for e in D.referents(program, names(unit, "y")[0])] == ["<argument m.s.y>"]
assert [str(e) for e in D.referents(program, names(unit, "v")[0])] == ["<variable m.s.v>"]
assert [str(e) for e in D.referents(program, names(unit, "q")[0])] == ["<argument m.p.q>"]
assert [str(e) for e in D.referents(program, names(unit, "n")[0])] == ["<argument m.twice.n>"]
assert [str(e) for e in D.referents(program, names(unit, "b")[0])] == ["<variable m.b>"]  # a clockvar's value is outside
assert [str(e) for e in D.referents(program, names(unit, "twice")[0])] == ["<let m.twice>"]
default = [n for n in walk(unit) if isinstance(n, S.DefaultClocking)][0]
assert [str(e) for e in D.referents(program, default.name)] == ["<clocking m.cb>"]
print("ok")


## VLGDEF-08 · Covergroups, their arguments, coverpoints, crosses, bins and filters

In [ ]:
unit, program = define("""class c; int a, b, en;
covergroup cg(int lo = 0, hi) @(posedge en);
  cp_a: coverpoint a iff (en) { bins low = {[0:lo]} with (item > b) iff (en); bins s = cp_a with (item < 3); }
  coverpoint b { bins one = {1}; }
  x: cross cp_a, b { bins both = binsof(cp_a.low) && binsof(b); function int f(); return lo; endfunction }
endgroup
covergroup sampled with function sample(int v); coverpoint v; endgroup
endclass""")
entities = [str(e) for e in program.entities()]
for entity in ["<covergroup c::cg>", "<argument c::cg::lo>", "<coverpoint c::cg::cp_a>", "<bins c::cg::cp_a::low>",
               "<bins c::cg::cp_a::s>", "<cross c::cg::x>", "<bins c::cg::x::both>", "<function c::cg::x::f>",
               "<covergroup c::sampled>", "<argument c::sampled::v>"]:
    assert entity in entities, entity
assert "<bins one>" in entities  # in an unlabeled coverpoint
assert [str(e) for e in D.referents(program, names(unit, "lo")[0])] == ["<argument c::cg::lo>"]
assert [str(e) for e in D.referents(program, names(unit, "item")[0])] == ["<variable item>"]
assert [str(e) for e in D.referents(program, names(unit, "b")[1])] == ["<variable c::b>"]  # in the filter
assert [str(e) for e in D.referents(program, names(unit, "cp_a")[0])] == ["<coverpoint c::cg::cp_a>"]
assert [str(e) for e in D.referents(program, names(unit, "lo")[-1])] == ["<argument c::cg::lo>"]  # in x's function
assert [str(e) for e in D.referents(program, names(unit, "v")[0])] == ["<argument c::sampled::v>"]
assert "<argument c::cg::hi>" in entities
assert [str(e) for e in D.referents(program, names(unit, "en")[-1])] == ["<variable c::en>"]  # a bin's condition
print("ok")


## VLGDEF-09 · `$unit::` names are found in the compilation unit

In [ ]:
unit, program = define("""parameter W = 1;
class c; static int k; endclass
module m; parameter W = 2;
initial x = W + $unit::W + $unit::c::k + $unit::missing;
endmodule""")
assert [str(e) for e in D.referents(program, names(unit, "W")[0])] == ["<parameter m.W>"]
outer = [n for n in walk(unit) if isinstance(n, S.ScopedName) and isinstance(n.scope, S.UnitName)]
assert [[str(e) for e in D.referents(program, n)] for n in outer] == [["<parameter W>"], ["<variable c::k>"], []]
print("ok")


## VLGDEF-10 · An enumeration's ranges declare numbered names; a net type is a type

In [ ]:
unit, program = define("""package p; typedef enum {PROBE[2], BAY[3:1] = 5, C['h2], E[N], D} e;
parameter N = 2;
nettype logic [1:0] level with resolve; function logic [1:0] resolve(); endfunction endpackage
module m; import p::*; initial x = PROBE1 + BAY2 + D; level w; endmodule""")
entities = [str(e) for e in program.entities()]
assert entities[1:7] == ["<enumerator p::PROBE0>", "<enumerator p::PROBE1>", "<enumerator p::BAY3>",
                         "<enumerator p::BAY2>", "<enumerator p::BAY1>", "<enumerator p::D>"], entities
assert "<type p::level>" in entities
assert [str(e) for e in D.referents(program, names(unit, "BAY2")[0])] == ["<enumerator p::BAY2>"]
members = unit.items[0].items[0].type.members
assert [str(e) for e in D.referents(program, members[0].name)] == ["<enumerator p::PROBE0>"]
assert D.referents(program, members[2].name) == D.referents(program, members[3].name) == []  # `C['h2]` and `E[N]`: not decimal numbers, no names
assert [str(e) for e in D.referents(program, unit.items[0].items[2].function)] == ["<function p::resolve>"]
print("ok")


## VLGDEF-11 · A non-ANSI port's references declare its ports; an explicit port's own name is not looked up

In [ ]:
unit, program = define("""module a (.x(x), .halves({lo, hi}), .first(lo[i]), , .unused());
input x; input [1:0] lo, hi; parameter i = 0;
endmodule
module b (input .sum(v + 1), output logic level); logic v; endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<module a>", "<port a.x>", "<port a.lo>", "<port a.hi>", "<parameter a.i>", "<module b>",
                    "<port b.level>", "<variable b.v>"], entities
explicit = [n for n in walk(unit) if isinstance(n, (S.ExplicitPort, S.ExplicitAnsiPort))]
assert all(D.referents(program, n.name) == [] for n in explicit)  # a port's own name is outside the module
assert [str(e) for e in D.referents(program, names(unit, "i")[0])] == ["<parameter a.i>"]
assert [str(e) for e in D.referents(program, names(unit, "v")[0])] == ["<variable b.v>"]
print("ok")


## VLGDEF-12 · An interface's type, a skipped `foreach` variable, and the macros of an `ifdef condition

In [ ]:
unit, program = define("""module n (bus_if bus); typedef bus.data_t t; logic [1:0] q [2][2]; logic A;
initial foreach (q[, j]) q[0][j] = A;
`ifdef (A && !B)
  initial x = 1;
`endif
endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<module n>", "<port n.bus>", "<type n.t>", "<variable n.q>", "<variable n.A>", "<variable j>"], entities
interface = [n for n in walk(unit) if isinstance(n, S.InterfaceTypeName)][0]
assert [str(e) for e in D.referents(program, interface.interface)] == ["<port n.bus>"]
assert D.referents(program, interface.name) == []  # a type in the interface: not resolved
condition = [i for i in unit.items[0].items if isinstance(i, S.IfdefDirective)][0].condition
assert [D.referents(program, n) for n in walk(condition) if isinstance(n, S.NameExpression)] == [[], []]  # macros
print("ok")


## VLGDEF-13 · An `extern` unit's ports, a DPI import's function, and what `bind` binds, found in its target

In [ ]:
unit, program = define("""extern module e #(parameter W = 1) (input a, output b);
module e (.*); assign b = a + W; endmodule
module m; import "DPI-C" pure c_f = function int f(int x); export "DPI-C" function f; mon k (); logic v; endmodule
bind m mon c (.a(v)); bind m: k mon2 c2 (); bind nowhere mon c3 (.a(w));
extern module g #(parameter W = 1) (input a); module g #(parameter V = 2) (.*); endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<module e>", "<parameter e.W>", "<port e.a>", "<port e.b>", "<module m>", "<function m.f>",
                    "<argument m.f.x>", "<instance m.k>", "<variable m.v>", "<module g>", "<parameter g.V>",
                    "<port g.a>"], entities  # the extern, and what bind binds,
# declare nothing
assert [str(e) for e in D.referents(program, names(unit, "W")[0])] == ["<parameter e.W>"]
export = [n for n in walk(unit) if isinstance(n, S.DpiExport)][0]
assert [str(e) for e in D.referents(program, export.name)] == ["<function m.f>"]
assert [str(e) for e in D.referents(program, names(unit, "v")[0])] == ["<variable m.v>"]  # in the target
assert [str(e) for e in D.referents(program, names(unit, "k")[0])] == ["<instance m.k>"]
assert D.referents(program, names(unit, "w")[0]) == [] and D.referents(program, names(unit, "nowhere")[0]) == []
print("ok")


## VLGDEF-14 · A `randsequence`'s productions, their ports, and the calls of them

In [ ]:
unit, program = define("""module m; int n;
initial randsequence (main)
  main : step(n) done;
  step(int n, k = 1) : { $display(n, k); };
  done : { };
endsequence
initial randsequence () only : { };
endsequence
endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<module m>", "<variable m.n>", "<production main>", "<production step>", "<argument step.n>",
                    "<argument step.k>", "<production done>", "<production only>"], entities
calls = [n for n in walk(unit) if isinstance(n, S.ProductionCall)]
assert [[str(e) for e in D.referents(program, c.name)] for c in calls] == [["<production step>"], ["<production done>"]]
sequence = unit.items[0].items[1].body
assert [str(e) for e in D.referents(program, sequence.first)] == ["<production main>"]
assert [[str(e) for e in D.referents(program, n)] for n in names(unit, "n")] == [["<variable m.n>"], ["<argument step.n>"]]
print("ok")


## VLGDEF-15 · Checkers' arguments and variables, and their instances, by name or in a package

In [ ]:
unit, program = define("""package p; checker pc (a); bit seen; assert property (a && seen); endchecker export *::*; endpackage
checker c (input logic clk, a); rand bit r; assert property (@(posedge clk) a |-> r); endchecker
module m; logic x; c c1 (.clk(x), .a(x)); always @(x) begin p::pc c2 (x); end endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<package p>", "<checker p::pc>", "<argument p::pc::a>", "<variable p::pc::seen>", "<checker c>",
                    "<argument c.clk>", "<argument c.a>", "<variable c.r>", "<module m>", "<variable m.x>",
                    "<instance m.c1>", "<instance c2>"], entities
assert [str(e) for e in D.referents(program, names(unit, "seen")[0])] == ["<variable p::pc::seen>"]
assert [str(e) for e in D.referents(program, names(unit, "r")[0])] == ["<variable c.r>"]
modules = [n.module for n in walk(unit) if isinstance(n, S.ModuleInstantiation)]
assert [[str(e) for e in D.referents(program, m)] for m in modules] == [["<checker c>"], ["<checker p::pc>"]]
print("ok")


## VLGDEF-16 · A pattern's variables, seen by its guard and body, or by a condition and its consequence

In [ ]:
unit, program = define("""module m; int v, w;
initial begin
  case (x) matches tagged a .v &&& v > 0: w = v; tagged b: w = v; endcase
  if (x matches tagged a .v &&& v > 1) w = v; else w = v;
  w = x matches '{.v, .w} ? v + w : v;
end
endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<module m>", "<variable m.v>", "<variable m.w>", "<variable v>", "<variable v>", "<variable v>",
                    "<variable w>"], entities
seen = [[str(e) for e in D.referents(program, n)] for n in names(unit, "v")]
assert seen == [["<variable v>"], ["<variable v>"], ["<variable m.v>"], ["<variable v>"], ["<variable v>"],
                ["<variable m.v>"], ["<variable v>"], ["<variable m.v>"]], seen  # the pattern's, then outside it
print("ok")


## VLGDEF-17 · A primitive's ports, its `initial` and `reg`, and its instances; named gates

In [ ]:
unit, program = define("""primitive latch (q, d, en);
output q; reg q; input d, en;
initial q = 1'b0;
table 1 1 : ? : 1 ; endtable
endprimitive
primitive dff (output reg q = 0, input d, clk); table 0 r : ? : 0 ; endtable endprimitive
module m; wire a, b, o; latch #1 (o, a, b); and g (o, a, b); not (o, a); endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<primitive latch>", "<port latch.q>", "<port latch.d>", "<port latch.en>", "<primitive dff>",
                    "<port dff.q>", "<port dff.d>", "<port dff.clk>", "<module m>", "<net m.a>", "<net m.b>", "<net m.o>",
                    "<instance m.g>"], entities  # an unnamed gate's or primitive's instance declares nothing
latch = unit.items[0]
assert [str(e) for e in D.referents(program, latch.initial.name)] == ["<port latch.q>"]
assert [str(e) for e in D.referents(program, latch.declarations[1].names[0])] == ["<port latch.q>"]  # `reg q;`
udp = unit.items[2].items[1]
assert [str(e) for e in D.referents(program, udp.primitive)] == ["<primitive latch>"]
assert [[str(e) for e in D.referents(program, n)] for n in udp.instances[0].terminals] == [["<net m.o>"], ["<net m.a>"],
                                                                                         ["<net m.b>"]]
print("ok")


## VLGDEF-18 · Specify parameters, and the ports and parameters paths and timing checks name

In [ ]:
unit, program = define("""module m (input a, clk, output y);
specparam [3:0] tStep = 1;
specify
  specparam tRise = 2, PATHPULSE$a$y = (1, 2);
  (a => y) = (tRise, tStep);
  $setup(a, posedge clk, tStep);
endspecify
endmodule""")
entities = [str(e) for e in program.entities()]
assert entities == ["<module m>", "<port m.a>", "<port m.clk>", "<port m.y>", "<specparam m.tStep>", "<specparam m.tRise>",
                    "<specparam m.PATHPULSE$a$y>"], entities
assert [[str(e) for e in D.referents(program, n)] for n in names(unit, "tStep")] == [["<specparam m.tStep>"]] * 2
path = [n for n in walk(unit) if isinstance(n, S.PathDeclaration)][0]
assert [[str(e) for e in D.referents(program, n)] for n in (*path.inputs, *path.outputs)] == [["<port m.a>"], ["<port m.y>"]]
print("ok")
